In [ ]:
!pip install gymnasium

In [ ]:
import gymnasium as gym
import numpy as np
import pandas as pd
import random
import joblib

In [ ]:
inventory = pd.read_csv("inventory.csv")
supplier = pd.read_csv("supplier.csv")
X_train = pd.read_csv("processed_features.csv")

model = joblib.load("xgboost_walmart.pkl")

In [ ]:
input_data = X_train.iloc[[0]].copy()

predicted_demand = model.predict(input_data)[0]

store = int(input_data["Store"].iloc[0])
dept = int(input_data["Dept"].iloc[0])

print(predicted_demand)

25672.266


In [ ]:
product = inventory[
    (inventory["Store"] == store) &
    (inventory["Dept"] == dept)
]

current_stock = int(product.iloc[0]["Current_Stock"])
lead_time = int(product.iloc[0]["Lead_Time"])
ordering_cost = float(product.iloc[0]["Ordering_Cost"])
holding_cost = float(product.iloc[0]["Holding_Cost"])

In [ ]:
class InventoryEnv(gym.Env):

    def __init__(self,
                 demand,
                 stock,
                 holding_cost,
                 ordering_cost):

        super().__init__()

        self.predicted_demand = demand
        self.current_stock = stock
        self.holding_cost = holding_cost
        self.ordering_cost = ordering_cost

        # Order quantities (units)
        self.actions = [0,200,400,600,800,1000]

    def reset(self, seed=None, options=None):

        self.stock = self.current_stock

        return np.array([self.stock],dtype=np.float32), {}

    def step(self, action):

        order_qty = self.actions[action]

        self.stock += order_qty

        sold = min(self.stock,self.predicted_demand)

        self.stock -= sold

        revenue = sold

        holding_penalty = self.stock * self.holding_cost

        ordering_penalty = order_qty * self.ordering_cost / 100

        stockout_penalty = max(
            0,
            self.predicted_demand - sold
        ) * 5

        reward = (
            revenue
            - holding_penalty
            - ordering_penalty
            - stockout_penalty
        )

        done = True

        return (
            np.array([self.stock],dtype=np.float32),
            reward,
            done,
            False,
            {}
        )

In [ ]:
env = InventoryEnv(
    predicted_demand,
    current_stock,
    holding_cost,
    ordering_cost
)

In [ ]:
max_stock = 5000

num_actions = 6

Q = np.zeros((max_stock+1,num_actions))
alpha = 0.1
gamma = 0.95
epsilon = 0.2

episodes = 1000

In [ ]:
for episode in range(episodes):

    state,_ = env.reset()

    state = int(min(state[0], max_stock))

    done = False

    while not done:

        if random.random() < epsilon:
            action = random.randint(0,5)
        else:
            action = np.argmax(Q[state])

        next_state,reward,done,_,_ = env.step(action)

        next_state = int(min(next_state[0], max_stock))

        Q[state,action] += alpha * (
            reward +
            gamma*np.max(Q[next_state]) -
            Q[state,action]
        )

        state = next_state

In [ ]:
state,_ = env.reset()

state = int(min(state[0], max_stock))

best_action = np.argmax(Q[state])

order_quantity = env.actions[best_action]

print("Best Action :", best_action)
print("Recommended Order :", order_quantity)

Best Action : 5
Recommended Order : 1000


In [ ]:
print("="*60)
print("AI INVENTORY OPTIMIZATION REPORT")
print("="*60)

print(f"Store               : {store}")
print(f"Department          : {dept}")
print(f"Predicted Demand    : {predicted_demand:.2f}")
print(f"Current Stock       : {current_stock}")
print(f"RL Recommended Qty  : {order_quantity}")
print(f"Lead Time           : {lead_time}")

AI INVENTORY OPTIMIZATION REPORT
Store               : 1
Department          : 1
Predicted Demand    : 25672.27
Current Stock       : 302
RL Recommended Qty  : 1000
Lead Time           : 6


In [ ]:
supplier.head()

,Supplier_ID,Supplier_Name,Cost_Per_Unit,Delivery_Time,Reliability
0,S001,ABC Traders,10,3,96
1,S002,Global Supply,9,5,90
2,S003,Prime Logistics,11,2,98
3,S004,Retail Distributors,8,4,88
4,S005,Smart Supply,10,3,94


In [ ]:
supplier["Supplier_Score"] = (
    0.5 * supplier["Reliability"]
    - 0.3 * supplier["Cost_Per_Unit"]
    - 0.2 * supplier["Delivery_Time"]
)

supplier

,Supplier_ID,Supplier_Name,Cost_Per_Unit,Delivery_Time,Reliability,Supplier_Score
0,S001,ABC Traders,10,3,96,44.4
1,S002,Global Supply,9,5,90,41.3
2,S003,Prime Logistics,11,2,98,45.3
3,S004,Retail Distributors,8,4,88,40.8
4,S005,Smart Supply,10,3,94,43.4


In [ ]:
supplier = supplier.sort_values(
    by="Supplier_Score",
    ascending=False
)

supplier

,Supplier_ID,Supplier_Name,Cost_Per_Unit,Delivery_Time,Reliability,Supplier_Score
2,S003,Prime Logistics,11,2,98,45.3
0,S001,ABC Traders,10,3,96,44.4
4,S005,Smart Supply,10,3,94,43.4
1,S002,Global Supply,9,5,90,41.3
3,S004,Retail Distributors,8,4,88,40.8


In [ ]:
best_supplier = supplier.iloc[0]

print(best_supplier)

Supplier_ID                  S003
Supplier_Name     Prime Logistics
Cost_Per_Unit                  11
Delivery_Time                   2
Reliability                    98
Supplier_Score               45.3
Name: 2, dtype: object
